# CTC Model Training Pipeline
This notebook implements the Connectionist Temporal Classification (CTC) pipeline. 
Unlike the sliding window approach, this trains the `CTC_CRNN` sequentially on entire audio recordings using PyTorch's native `CTCLoss`.

In [13]:
import sys

assert sys.version_info >= (3, 10)
IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    !git clone https://github.com/stachuapa123/ASR_project.git
    %cd ASR_project
    # !git checkout <YOUR_BRANCH_NAME>  # Uncomment and set this to your branch if needed
    !pip install -q torchmetrics
    from google.colab import drive

    drive.mount("/content/drive")

    # Extract data securely if on Colab
    !mkdir -p "/content/asr_data"
    !unzip -q "/content/drive/MyDrive/asr_data.zip" -d "/content/asr_data"
    DATA_DIR = "/content/asr_data"
else:
    # Local path
    %load_ext autoreload
    %autoreload 2
    DATA_DIR = "../data"  # Update to your local subset or AutorskieDane

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [14]:
import torch
from torch.utils.data import DataLoader

from src.ctc.config import CTCConfig as C
from src.ctc.model import CTCModel
from src.ctc.dataset import CTCDataset, ctc_collate_fn
from src.ctc.augmentation import SpecAugment
from src.ctc.training import train_ctc, EarlyStopping

In [ ]:
# Hyperparameters
PCT_VAL = 0.15
BATCH_SIZE = 16
N_EPOCHS = 100
LR = 1e-3
MAX_LR = 3e-3
WEIGHT_DECAY = 1e-4
PCT_START = 0.2
NUM_WORKERS = 4

device = C.get_device()
print(f"Using device: {device}")

Using device: cuda


In [ ]:
dataset = CTCDataset(data_root=DATA_DIR, cache_mode=False, apply_augmentations=True)

n_total = len(dataset)
n_val = max(1, int(0.15 * n_total))
n_train = n_total - n_val
generator = torch.Generator().manual_seed(42)
train_set, val_set = torch.utils.data.random_split(
    dataset,
    [n_train, n_val],
    generator=generator,
)
print(f"Train items: {len(train_set)} | Val items: {len(val_set)}")

train_loader = DataLoader(
    train_set,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=ctc_collate_fn,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)
val_loader = DataLoader(
    val_set,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=ctc_collate_fn,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

Train items: 4 | Val items: 1


In [ ]:
model = CTCModel()
objective = torch.nn.CTCLoss(blank=C.BLANK_IDX, zero_infinity=True)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=MAX_LR,
    steps_per_epoch=len(train_loader),
    epochs=N_EPOCHS,
    pct_start=PCT_START,
)
scaler = torch.amp.GradScaler(
    device=device.type,
    enabled=(device.type == "cuda"),
)
es = EarlyStopping()
spec_augment = SpecAugment()

In [ ]:
model = train_ctc(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    objective=objective,
    device=device,
    n_epochs=N_EPOCHS,
    spec_augment=spec_augment,
    scheduler=scheduler,
    scaler=scaler,
    early_stopping=es,
    save_best_to="../trained_models/ctc_test.pt",
    use_amp=(device.type == "cuda"),
    step_scheduler_per_batch=True,
)

Epoch   1/3 | Train Loss: 6.5604 | Val Loss: 5.8434 | Val PER: 0.8923 | LR: 1.4e-04 [BEST]          
Epoch   2/3 | Train Loss: 6.4617 | Val Loss: 5.8264 | Val PER: 0.9692 | LR: 2.0e-04          
Epoch   3/3 | Train Loss: 6.3495 | Val Loss: 5.7983 | Val PER: 0.9846 | LR: 2.9e-04          
Restored best weights with Val PER = 0.8923
